# This file is for experimenting and seeing what works and how. All the final code will be available in main.py.
- Now I'm using the simplegmail library to parse mails.
- Wish me luck monekeys.

In [58]:
from simplegmail import Gmail

In [59]:
gmail = Gmail(creds_file="token.pickle", client_secret_file="credentials.json")

In [60]:
messages = gmail.get_messages(query = "newer_than:90d")

In [61]:
len(messages)

289

In [62]:
mtypes = set()
for message in messages:
    mtypes.add(message.headers["Content-Type"].split(";")[0]) # get mtypes. 
# this library returns content type as a string, such that:
# "content-type: multipart/...; boundary: xxx". so when we split at semicolon, and take first element of the list,
# we get content type yessir ^^

In [63]:
mtype_classified = {m: [] for m in mtypes} # dict for classifying

In [64]:
for message in messages:
    mtype_classified[message.headers["Content-Type"].split(";")[0]].append(message)

In [65]:
mtype_classified["multipart/related"] # example

[Message(to: undisclosed-recipients:;, from: Elliott P <elliott@blueprint.hackclub.com>, id: 19e2e2cbf90369b7)]

In [66]:
message_count = 0

for key in mtype_classified.keys():
    for message in mtype_classified[key]:
        message_count += 1

In [67]:
message_count

289

### ***Nice, it's working.***

btw im borrowing this plan from the previous notebook.

***Data Collection***
- Collect message ID 
- Collect sender name 
- Collect sender domain, sender local part 
- Collect combined text(subject + body) 
- Collect word count of body 
- Collect number of numerical characters(digits) in body. 
- Collect link count(useful for promotional mails or maybe school mails? idk) 
- Collect currency count(useful for promotional i think) 
- Collect presence of otp-like numbers 
- Collect exclamation mark count(spammy emails) 
- Also record contains_html, and contains_plain.
- Also record presence of html structures like image_count, button_count, etc...

***Final Dataframe***
- Sender info 
- Combined text
- Word count
- Digit count
- Mail folder
- Message ID(DO NOT TRAIN ON THIS)

***Marking Targets***
- Perform unsupervised KMeans to get clusters.

***Model and Final pipeline***
- Gaussian naive bayes.

In [99]:
"""

- Collect message ID ✅
- Collect sender name ✅
- Collect sender domain, sender local part ✅
- Collect combined text(subject + body) ✅
- Collect word count of body ✅
- Collect number of numerical characters(digits) in body. ✅
- Collect link count(useful for promotional mails or maybe school mails? idk) ✅
- Collect currency count(useful for promotional i think) ✅
- Collect presence of otp-like numbers ✅
- Collect exclamation mark count(spammy emails) ✅
- Also record contains_html, and contains_plain. ✅

"""

from email.utils import parseaddr
import html, re
from bs4 import BeautifulSoup

# function to clean mails of special chars

def cleaner(text):
    cleaned = html.unescape(text)
    cleaned = re.sub(r'[\u200c\u200b\u200d\ufeff\xad]', "", cleaned) 
    # [] => match any one char, an since every char is unicode, regex treats \u200c(for eg) as a single char
    cleaned = " ".join(cleaned.split())
    return cleaned

# quick function to use expressions based on which type of msg we have

def parse_mail(msg):
    msg_id = msg.id
    sender = msg.sender
    sender_name, sender_addr = parseaddr(sender)
    sender_local, sender_domain = sender_addr.rsplit("@", 1)
    subject = cleaner(msg.subject)

    contains_html, contains_plain = bool(msg.html), bool(msg.plain)

    soup = BeautifulSoup(msg.html, "html.parser") if contains_html else "" # for html

    body = re.sub(r"https?://S+", "",cleaner(msg.plain)) if not contains_html else cleaner(soup.get_text(strip = True, separator = " "))# to remove links
    combined_text = f"{subject} {body}"

    word_count = len(body.split())

    digit_count = sum([char.isdigit() for char in body])

    link_count = len(soup.find_all("a", href=True)) if contains_html else re.findall(r"https?://S+", body)

    currency_count = sum([char in ["₹", "$"] for char in body])

    otp_count = len(re.findall(r"\b\d{4,8}\b", body))

    exclamation_count = sum([char == "!" for char in body])

    image_count = 0 if not contains_html else len(soup.find_all("img"))

    # final record

    record = {"id": msg_id,
              "contains_html": contains_html,
              "contains_plain_text": contains_plain,
              "sender_name": sender_name,
              "sender_local": sender_local,
              "sender_address": sender_addr,
              "comb_text": combined_text,
              "word_count": word_count,
              "digit_count": digit_count,
              "link_count": link_count,
              "currency_count": currency_count,
              "otp_count": otp_count,
              "exclamation_count": exclamation_count,
              "image_count": image_count
              }
    
    return record

In [100]:
print([message for message in messages if not message.html and message.plain][0].html)

None


In [101]:
records = [parse_mail(message) for message in messages][:]


In [104]:
import pandas as pd

df = pd.DataFrame(records).set_index("id")

In [105]:
df.head()

,contains_html,contains_plain_text,sender_name,sender_local,sender_address,comb_text,word_count,digit_count,link_count,currency_count,otp_count,exclamation_count,image_count
id,,,,,,,,,,,,,
19ea10ec651d6d89,True,True,Muhammad Rihaan,syedmrihaan,syedmrihaan@gmail.com,Re: Revised Offer Letter for AIYA and Confirma...,1124,95,5,0,11,0,2
19ea0f19336a3569,True,True,Nathan from Macondo (Hack Club),macondo,macondo@hackclub.com,"Your ticket ""hey guys quick question do we get...",237,1,2,0,0,0,1
19e9dd2429d2da77,True,False,ChatGPT,noreply,noreply@email.openai.com,Action Required: Important security update for...,246,40,11,0,7,0,5
19e99542404b66d0,True,True,Hack Club,identity,identity@hackclub.com,[Hack Club] Your identity verification has bee...,58,4,1,0,0,3,1
19e98d4b48ef2d88,True,True,RUR Greenlife,info,info@rur.co.in,Happy World Environment Day 2026! Dear Greenie...,112,12,9,0,0,2,6
